In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [2]:
!kaggle datasets download -d adiithape1/plant-disease-detection-dataset-master-version

Dataset URL: https://www.kaggle.com/datasets/adiithape1/plant-disease-detection-dataset-master-version
License(s): CC-BY-SA-4.0
100% 7.69G/7.69G [01:34<00:00, 87.3MB/s]



In [3]:
import zipfile
zip_ref = zipfile.ZipFile('/content/plant-disease-detection-dataset-master-version.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [4]:
!pip install -q tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, regularizers, layers
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, BatchNormalization, Dropout, GlobalAveragePooling2D
from keras.src.callbacks.monitor_callback import MonitorCallback
from PIL import Image

In [5]:
batch_size = 64

train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    '/content/MasterDataset/train',
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='sparse')

validation_generator = test_datagen.flow_from_directory(
    '/content/MasterDataset/val',
    target_size=(150, 150),
    batch_size=batch_size,
    class_mode='sparse')

Found 115076 images belonging to 86 classes.
Found 14356 images belonging to 86 classes.


In [6]:
#generators
train_dataset = keras.utils.image_dataset_from_directory(
    directory='MasterDataset/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150)
)

validation_dataset = keras.utils.image_dataset_from_directory(
    directory='MasterDataset/val',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150)
)

Found 115076 files belonging to 86 classes.
Found 14356 files belonging to 86 classes.


In [7]:
#normalize
def process(image, label):
    image = tf.cast(image/255. , tf.float32)
    return image, label

train_dataset = train_dataset.map(process)
validation_dataset = validation_dataset.map(process)

In [8]:
#cnn model
model = Sequential()

model.add(Conv2D(64, kernel_size = (3, 3), activation = "relu", input_shape = (150, 150, 3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2, 2), strides = 2, padding = "valid"))
model.add(Dropout(0.25))

model.add(Conv2D(128, kernel_size = (3, 3), activation = "relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2, 2), strides = 2, padding = "valid"))
model.add(Dropout(0.25))

model.add(Conv2D(128, kernel_size = (3, 3), activation = "relu"))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size = (2, 2), strides = 2, padding = "valid"))
model.add(Dropout(0.25))

model.add(GlobalAveragePooling2D())
model.add(Flatten())
model.add(BatchNormalization())

model.add(Dense(128, activation = "relu", kernel_regularizer=regularizers.l2(1e-4)))
model.add(Dropout(0.4))

model.add(Dense(128, activation = "relu", kernel_regularizer=regularizers.l2(1e-4)))
model.add(Dropout(0.4))

model.add(Dense(86, activation = "softmax"))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#compile
reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
callback =  keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

model.compile(optimizer = "Adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])


history = model.fit(
    train_generator,
    callbacks=[reduce_lr, callback],
    steps_per_epoch = train_generator.samples // batch_size,
    validation_data = validation_generator,
    validation_steps = validation_generator.samples // batch_size,
    epochs = 25)

model.summary()

Epoch 1/25
 301/1798 ━━━━━━━━━━━━━━━━━━━━ 14:18 573ms/step - accuracy: 0.1681 - loss: 3.2734

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [11]:
# Check available GPUs
gpus = tf.config.list_physical_devices("GPU")

if not gpus:
    raise RuntimeError(
        "No GPU detected. Check nvidia-smi and CUDA/cuDNN installation."
    )

try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth set successfully.")
except RuntimeError as e:
    print("Could not set memory growth (already initialized):", e)

print("Using GPU:", gpus)

Could not set memory growth (already initialized): Physical devices cannot be modified after being initialized
Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
